
# ✅ Stretch Solutions — NumPy ndarrays & Vectorization Lab

This notebook contains worked solutions for the Stretch items:
1. Add a third feature (`sleep_hours`) and re-run predictions.
2. Standardize features and re-run.
3. Implement a reusable `accuracy(X, y, w, b)` helper.
4. Scale to a larger dataset and compare loop vs. vectorized timings.


In [6]:

import numpy as np
import time

rng = np.random.default_rng(123)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def accuracy(X, y, w, b, thresh: float = 0.5) -> float:
    z = X @ w + b
    z = np.clip(z, -50.0, 50.0)  # prevents exp overflow
    probs = 1.0 / (1.0 + np.exp(-z))
    y_pred = (probs >= thresh).astype(np.int64)
    return (y_pred == y).mean()

# Base dataset (same as lab)
n_samples = 500
study_hours = rng.normal(loc=5, scale=2, size=n_samples)
attendance_rate = rng.uniform(low=0.5, high=1.0, size=n_samples)
X = np.column_stack([study_hours, attendance_rate])

linear_combo = 0.6 * X[:, 0] + 0.4 * X[:, 1] * 10
noise = rng.normal(0, 0.5, size=n_samples)
scores_true = linear_combo + noise
y = (scores_true > 4.5).astype(np.int64)

# Base weights as per lab
w_base = np.array([0.8, 1.5])
b_base = -4.0

X.shape, y.shape


((500, 2), (500,))


## SG1 — Third feature (`sleep_hours`) and re-run


In [7]:

sleep_hours = rng.normal(loc=7, scale=1.5, size=n_samples)
X3 = np.column_stack([study_hours, attendance_rate, sleep_hours])

w3 = np.array([0.8, 1.5, 0.2])
b3 = -4.0

probs3 = sigmoid(X3 @ w3 + b3)
y_pred3 = (probs3 >= 0.5).astype(np.int64)
acc3 = (y_pred3 == y).mean()

X3.shape, w3.shape, acc3


((500, 3), (3,), np.float64(0.928))


## SG2 — Standardize features and re-run


In [8]:

# 2-feature standardization
mu2 = X.mean(axis=0); std2 = X.std(axis=0) + 1e-8
X_std2 = (X - mu2) / std2
acc_std2 = accuracy(X_std2, y, w_base, b_base)

# 3-feature standardization
mu3 = X3.mean(axis=0); std3 = X3.std(axis=0) + 1e-8
X_std3 = (X3 - mu3) / std3
acc_std3 = accuracy(X_std3, y, w3, b3)

acc_std2, acc_std3


(np.float64(0.128), np.float64(0.128))


## SG3 — Accuracy helper quick checks


In [9]:

acc_a = accuracy(X, y, w_base, b_base)
acc_b = accuracy(X_std2, y, w_base, b_base)

assert 0.0 <= acc_a <= 1.0
assert 0.0 <= acc_b <= 1.0

acc_all_one = accuracy(X, y, np.zeros_like(w_base),  9999.0)
acc_all_zero = accuracy(X, y, np.zeros_like(w_base), -9999.0)

(acc_a, acc_b, acc_all_one, acc_all_zero)


(np.float64(0.876), np.float64(0.128), np.float64(0.874), np.float64(0.126))


## SG4 — Large n_samples timing comparison


In [10]:

n_big = 500_000
study_hours_big = rng.normal(loc=5, scale=2, size=n_big)

# Loop
t0 = time.perf_counter()
total = 0.0
for h in study_hours_big:
    total += h
avg_loop = total / len(study_hours_big)
t1 = time.perf_counter()
loop_ms = (t1 - t0) * 1e3

# Vectorized
t0 = time.perf_counter()
avg_vec = np.mean(study_hours_big)
t1 = time.perf_counter()
vec_ms = (t1 - t0) * 1e3

avg_loop, avg_vec, loop_ms, vec_ms, f"Speedup ≈ {loop_ms/vec_ms:0.1f}×"


(np.float64(5.000236955097732),
 np.float64(5.0002369550978525),
 107.71163800018257,
 0.38327999936882406,
 'Speedup ≈ 281.0×')